In [1]:
import pandas as pd
import numpy as np
from sklearn.ensemble import RandomForestRegressor
from sklearn.ensemble import ExtraTreesRegressor
from sklearn.model_selection import train_test_split, cross_val_score, GridSearchCV
from quantile_forest import ExtraTreesQuantileRegressor

import rasterio
import joblib
import time

# ---------------------------------------------------------------------------------------------
# Load table and light data wrangling
lsms_spatial = pd.read_csv('../data/processed/lsms_spatial_with_country_names.csv')     

# lsms_spatial = lsms_spatial[~ lsms_spatial['country'] .isin (['Ghana', 'Rwanda'])]
lsms_spatial = lsms_spatial[['farm_area_ha', 'cropland', 'cattle', 'pop', 'cropland_per_capita',
       'sand', 'slope', 'temperature', 'rainfall', 'maizeyield', 'market']]
lsms_spatial = lsms_spatial.dropna()
# lsms_spatial = lsms_spatial[:1000] # uncomment to run script faster
print(lsms_spatial.columns)
print(lsms_spatial)

# define input and output
X = lsms_spatial.drop(columns = ['farm_area_ha'])
y = lsms_spatial['farm_area_ha']
# -----------------------------------


# Quantile Extra-Trees Regressor model
print('---------------------Quantile Extra-Trees Regressor------------------------------')
start_time = time.time()

# Initialize the Quantile Extra-Trees Regressor
qrf = ExtraTreesQuantileRegressor(
    n_estimators = 1500,
    min_samples_split = 49,
    min_samples_leaf = 5,
    max_features = 8,
    oob_score = True, 
    bootstrap = True, 
    random_state = 2024
)

## Perform cross-validation to get CV R-squared  # needs a lot of memory!
# cv_scores_qrf = cross_val_score(qrf, X, y, cv=10, scoring='r2', n_jobs=-1)
# cv_r2_qrf = cv_scores_qrf.mean()
# print("Quantile Extra-Trees CV R-squared: ", cv_r2_qrf)

# Fit the model to get OOB R-squared
qrf.fit(X, y)
oob_r2_qrf = qrf.oob_score_
end_time = time.time()

print(f"Quantile Extra-Trees training time: {end_time - start_time} seconds")
print("Quantile Extra-Trees OOB R-squared: ", oob_r2_qrf)


# -------------------------------------------------------------------------
# Predict on input raster
# Load the input raster
input_file = '../data/processed/stacked_rasters_africa.tif'
with rasterio.open(input_file) as src:
    input_raster = src.read()  # Read all bands
    profile = src.profile

# Reshape the raster data for prediction
n_bands, height, width = input_raster.shape
input_raster_reshaped = input_raster.reshape(n_bands, -1).T  # Reshape to (n_samples, n_features)

# Filter out rows with NaN values
valid_mask = ~np.isnan(input_raster_reshaped).any(axis=1)
input_raster_valid = input_raster_reshaped[valid_mask]

# Predict using the loaded QRF model
quantiles = np.arange(0.01, 1.01, 0.01).tolist()  # Convert to list
qrf_output_valid = qrf.predict(input_raster_valid, quantiles = quantiles)

# Create an output array for QRF and fill with NaNs
qrf_output_raster = np.full((height * width, qrf_output_valid.shape[1]), np.nan)
qrf_output_raster[valid_mask] = qrf_output_valid
qrf_output_raster = qrf_output_raster.reshape(height, width, qrf_output_valid.shape[1])

# Write the QRF output raster
output_qrf_file = '../data/processed/qrf_100quantiles_predictions_africa.tif'
profile.update(count=qrf_output_raster.shape[2])  # Update profile for multiple bands
with rasterio.open(output_qrf_file, 'w', **profile) as dst:
    for i in range(qrf_output_raster.shape[2]):
        dst.write(qrf_output_raster[:, :, i], i + 1)


Index(['farm_area_ha', 'cropland', 'cattle', 'pop', 'cropland_per_capita',
       'sand', 'slope', 'temperature', 'rainfall', 'maizeyield', 'market'],
      dtype='object')
        farm_area_ha     cropland        cattle         pop  \
0           0.095855  1562.800049    787.684998   79.315643   
1           2.000000  1562.800049    787.684998   79.315643   
2           0.223116  1562.800049    787.684998   79.315643   
3           3.064620  1562.800049    787.684998   79.315643   
4           0.141745  1562.800049    787.684998   79.315643   
...              ...          ...           ...         ...   
166549      0.457500  4146.000000   1945.512085   54.440788   
166550      0.493900  4146.000000   1945.512085   54.440788   
166551      0.263200  4146.000000   1945.512085   54.440788   
166552      0.170000  4146.000000   1945.512085   54.440788   
166553      0.404858  2234.000000  16306.639648  111.792442   

        cropland_per_capita       sand     slope  temperature     rain

C:\Users\DHOUGNI\AppData\Local\Programs\Python\Python313\Lib\site-packages\sklearn\base.py:493: UserWarning: X does not have valid feature names, but ExtraTreesQuantileRegressor was fitted with feature names
  warnings.warn(


In [13]:
import pandas as pd
import numpy as np
from PyGRF import PyGRF
from sklearn.model_selection import train_test_split

import rasterio
import joblib
import time

# ---------------------------------------------------------------------------------------------
# Load table and light data wrangling
lsms_spatial = pd.read_csv('../data/processed/lsms_spatial_with_country_names.csv')     # this is with the avg of 3 cropland rasters

# lsms_spatial = lsms_spatial[~ lsms_spatial['country'] .isin (['Ghana', 'Rwanda'])]
lsms_spatial = lsms_spatial[['farm_area_ha', 'cropland', 'cattle', 'pop', 'cropland_per_capita',
       'sand', 'slope', 'temperature', 'rainfall', 'maizeyield', 'market']]
lsms_spatial = lsms_spatial.dropna()
lsms_spatial = lsms_spatial[:100] # uncomment to run script faster
print(lsms_spatial.columns)
print(lsms_spatial)

# define input and output
X = lsms_spatial.drop(columns = ['farm_area_ha'])
y = lsms_spatial['farm_area_ha']
# -----------------------------------

# Split data into training and test sets
X_train, X_test, y_train, y_test = train_test_split(X, y, random_state = 2024)

#Create a PyGRF model by specifying hyperparameters
pygrf_example = PyGRF.PyGRFBuilder(
    n_estimators = 60,
    max_features = 3,
    band_with = 50,
    train_weighted = true,
    predict_weighted = true,
    bootstrap = False,
    resample = True,
    random_state =2024
)

#Fit the created PyGRF model based on training data and their spatial coordinates
#xy_coord is the two-dimensional coordinates of training samples						  
pygrf_example.fit(X_train, y_train, xy_coord)

#Make predictions for testing data using the fitted PyGRF model and you specified local model weight 
predict_combined, predict_global, predict_local = pygrf_example.predict(X_test, coords_test, local_weight = 0.46)
print(pygrf_example)


ModuleNotFoundError: No module named 'PyGRF'

In [12]:
!pip install pyGRF

Defaulting to user installation because normal site-packages is not writeable
  Preparing metadata (setup.py): started
  Preparing metadata (setup.py): finished with status 'done'
  Obtaining dependency information for libpysal from https://files.pythonhosted.org/packages/fd/b0/744e3d450813c645cd092147bd72a7bdec0e3f4c1c4f5e3ff663252c7027/libpysal-4.12.1-py3-none-any.whl.metadata
  Obtaining dependency information for esda from https://files.pythonhosted.org/packages/a0/1b/84eaa84fa0e2b56464665f1d2135e0afe8ab1df481e6aa1fcdcb480032d6/esda-2.6.0-py3-none-any.whl.metadata
   ---------------------------------------- 0.0/135.4 kB ? eta -:--:--
   --------- ------------------------------ 30.7/135.4 kB 1.3 MB/s eta 0:00:01
   ---------------------------------------- 135.4/135.4 kB 2.0 MB/s eta 0:00:00
   ---------------------------------------- 0.0/2.8 MB ? eta -:--:--
   ---- ----------------------------------- 0.3/2.8 MB 5.9 MB/s eta 0:00:01
   ------- -------------------------------- 0.5/2.

In [1]:
pip list

Package                   VersionNote: you may need to restart the kernel to use updated packages.

------------------------- --------------
affine                    2.4.0
anyio                     4.6.2.post1
argon2-cffi               23.1.0
argon2-cffi-bindings      21.2.0
arrow                     1.3.0
asttokens                 3.0.0
async-lru                 2.0.4
attrs                     24.2.0
babel                     2.16.0
beautifulsoup4            4.12.3
bleach                    6.2.0
build                     1.2.2.post1
certifi                   2024.8.30
cffi                      1.17.1
charset-normalizer        3.4.0
click                     8.1.7
click-plugins             1.1.1
cligj                     0.7.2
colorama                  0.4.6
comm                      0.2.2
debugpy                   1.8.9
decorator                 5.1.1
defusedxml                0.7.1
exceptiongroup            1.2.2
executing                 2.1.0
fastjsonschema            2.21.1
fqdn